# Spanish Transcritption

In [ ]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.9 MB/s eta 0:00:00


In [ ]:
!pip install pyannote.audio==3.1.1 --upgrade

In [ ]:
!pip install faster_whisper

In [ ]:
!pip install pyannote.audio


In [ ]:
!pip install noisereduce

## Imports


In [ ]:
import numpy as np
import torch
import noisereduce as nr
import soundfile as sf
from faster_whisper import WhisperModel
from pydub import AudioSegment
import tempfile, os

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## Load and Resample

In [ ]:
def load_and_resample(path: str, target_sr: int = 16000):
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max   # normalize to [-1, 1]
    return samples, target_sr

## VAD

In [ ]:
def run_vad(samples: np.ndarray, sr: int):
    model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False
    )
    (get_speech_timestamps, _, read_audio, *_) = utils

    audio_tensor = torch.FloatTensor(samples)
    speech_timestamps = get_speech_timestamps(
        audio_tensor, model,
        sampling_rate=sr,
        threshold=0.5,
        min_speech_duration_ms=250,
        min_silence_duration_ms=500,
    )
    segments = [(t["start"] / sr, t["end"] / sr) for t in speech_timestamps]
    print(f"[VAD] Found {len(segments)} speech segments")
    return segments

## Concatenate only the VAD speech chunks

In [ ]:
def extract_speech_audio(samples: np.ndarray, sr: int, segments):
    chunks = [samples[int(s * sr): int(e * sr)] for s, e in segments]
    speech_audio = np.concatenate(chunks) if chunks else samples
    print(f"[VAD] Kept {len(speech_audio)/sr:.1f}s of speech "
          f"from {len(samples)/sr:.1f}s total")
    return speech_audio

## Noise Reduction

In [ ]:
def reduce_noise(samples: np.ndarray, sr: int):
    """Spectral noise reduction (noisereduce)."""
    denoised = nr.reduce_noise(y=samples, sr=sr, stationary=False, prop_decrease=0.8)
    print("[Noise Reduction] Done")
    return denoised

## Transcribe

In [ ]:
def transcribe(samples: np.ndarray, sr: int, speech_segments: list):
    model = WhisperModel("medium", device="cuda", compute_type="float16")

    results = []
    for i, (start, end) in enumerate(speech_segments):
        chunk = samples[int(start * sr): int(end * sr)]

        # Skip very short chunks
        if len(chunk) / sr < 0.5:
            continue

        segments, info = model.transcribe(
            chunk,
            language= "es",
            beam_size=5,
            vad_filter=False,
        )

        for seg in segments:
            results.append({
                "start": start + seg.start,   # offset by chunk start
                "end":   start + seg.end,
                "text":  seg.text.strip(),
                "lang":  info.language,
            })
            print(f"  [{start + seg.start:.1f}s → start + {seg.end:.1f}s] "
                  f"[{info.language}] {seg.text.strip()}")

    return results

## Joining Transcription

In [ ]:
def join_transcript(results):
   full_transcript = " ".join(r["text"] for r in results)


## Post Processing

### Alignment

In [ ]:
TARGET_SENTENCES = [
"Quiero cortarme el pelo",
"El libro está en la mesa",
"El carro lo tiene Pedro",
"El se ducha cada mañana",
"¿Qué dice usted que va a hacer hoy?",
"Dudo que sepa manejar muy bien",
"Las calles de esta ciudad son muy anchas",
"Puede que llueva mañana todo el día",
"Las casas son muy bonitas pero caras",
"Me gustan las películas que acaban bien",
"El chico con el que yo salgo es español",
"Después de cenar me fui a dormir tranquilo",
"Quiero una casa en la que vivan mis animales",
"A nosotros nos fascinan las fiestas grandiosas",
"Ella sólo bebe cerveza y no come nada",
"Me gustaría que el precio de las casas bajara",
"Cruza a la derecha y después sigue todo recto",
"Ella ha terminado de pintar su apartamento",
"Me gustaría que empezara a hacer más calor pronto",
"El niño al que se le murió el gato está triste",
"Una amiga mía cuida a los niños de mi vecino",
"El gato que era negro fue perseguido por el perro",
"Antes de poder salir él tiene que limpiar su cuarto",
"La cantidad de personas que fuman ha disminuido",
"Después de llegar a casa del trabajo tomé la cena",
"El ladrón al que atrapó la policía era famoso",
"Le pedí a un amigo que me ayudara con la tarea",
"El examen no fue tan difícil como me habían dicho",
"¿Serías tan amable de darme el libro que está en la mesa?",
"Hay mucha gente que no toma nada para el desayuno"
]

In [ ]:
import Levenshtein

def align_with_target(pred, target):
    distance = Levenshtein.distance(pred, target)
    ratio = Levenshtein.ratio(pred, target)
    return distance, ratio

In [ ]:
def evaluate_alignment(results, targets):
    evaluations = []

    for i, target in enumerate(targets):
        if i < len(results):
            pred = results[i]["text"]
        else:
            pred = ""

        dist, ratio = align_with_target(pred, target)

        evaluations.append({
            "sentence_id": i+1,
            "target": target,
            "prediction": pred,
            "distance": dist,
            "similarity": ratio
        })

    return evaluations

## Pipeline

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
AUDIO_PATH     = "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3"   # any format
TARGET_SR      = 16000              # Whisper expects 16kHz
HF_TOKEN       = ""         # for pyannote VAD
TRANSCRIBE_LANG = None                    # None = auto-detect; "es" = force Spanish
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 1. Load + resample
    audio, sr = load_and_resample(AUDIO_PATH, TARGET_SR)

    # 2. VAD
    speech_segments = run_vad(audio, sr)

    # 3. Noise reduce full audio (before chunking)
    audio = reduce_noise(audio, sr)

    results = transcribe(audio, sr, speech_segments)

    print("\n─── SENTENCE-WISE TRANSCRIPT ───")
    for i, r in enumerate(results):
      print(f"{i+1}. {r['text']}")

    full_transcript = " ".join(r["text"] for r in results)

    print("\n─── TRANSCRIPT ───")
    print(full_transcript)

    evaluations = evaluate_alignment(results, TARGET_SENTENCES)

    for e in evaluations:
      print(f"\nSentence {e['sentence_id']}")
      print("Target:     ", e["target"])
      print("Prediction: ", e["prediction"])
      print(f"Similarity: {e['similarity']:.3f}")

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 46 speech segments
[Noise Reduction] Done
  [74.1s → start + 1.3s] [es] nos fuimos al parque.
  [81.6s → start + 2.0s] [es] la llamaré mañana a la noche.
  [89.2s → start + 2.0s] [es] Puedes comprar carne en la tienda de Butcher.
  [97.5s → start + 2.4s] [es] Mi hermano recibió un nuevo computador.
  [107.7s → start + 2.6s] [es] A veces toman a su papá para una caminada en el parque.
  [118.9s → start + 3.2s] [es] Vamos a jugar a la voleybal en el gimnasio que os he dicho.
  [165.3s → start + 2.0s] [es] Quiero cortar mi alpelo
  [172.6s → start + 1.6s] [es] El libro está en la mesa.
  [180.0s → start + 2.0s] [es] El caro lo tiene pelo.
  [188.1s → start + 1.6s] [es] ese duché cada mañana.
  [197.2s → start + 2.4s] [es] ¿Qué dices ustedes que van a hacer hoy?
  [206.3s → start + 2.4s] [es] Dudo que sepa manejar muy bien.
  [216.1s → start + 3.0s] [es] Las calles de esta ciudad son muy anchas.
  [227.5s → start + 2.6s] [es] puede que lleve mañana toda el día.
  [238.7s → star